### Import

In [1]:
import numpy as np
import pandas as pd
import datetime as dt

pd.options.display.max_rows = 1000
pd.options.display.max_colwidth = 10000
pd.set_option('display.max_columns', 500)

### general parameters

In [2]:
mimiciii = "PATH TO DATA/mimic-iii(1.4)/"

### patients

In [3]:
patients = pd.read_csv(mimiciii + "PATIENTS.csv")

In [4]:
patients['DOD'] = pd.to_datetime(patients['DOD'])
patients.drop(columns=['ROW_ID', 'DOD_HOSP', 'DOD_SSN'], inplace=True)

In [5]:
patients.head(3)

In [6]:
print(patients.SUBJECT_ID.nunique())

46520


### admission

In [7]:
admissions = pd.read_csv(mimiciii + "ADMISSIONS.csv")

In [8]:
admissions = admissions[['SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'DISCHTIME', 'DEATHTIME', 'ETHNICITY', 
                         'DIAGNOSIS', 'HOSPITAL_EXPIRE_FLAG']]

In [9]:
admissions['ADMITTIME'] = pd.to_datetime(admissions['ADMITTIME'])
admissions['DISCHTIME'] = pd.to_datetime(admissions['DISCHTIME'])
admissions['DEATHTIME'] = pd.to_datetime(admissions['DEATHTIME'])

In [10]:
admissions.head(3)

In [11]:
print(admissions.SUBJECT_ID.nunique())
print(admissions.HADM_ID.nunique())

46520
58976


### merge admission and patients

In [12]:
cohort_df = pd.merge(admissions, patients, on=['SUBJECT_ID'], how='left')

In [13]:
cohort_df['DEATH_TIME_DISCH'] = cohort_df['DOD'] - cohort_df['DISCHTIME']
cohort_df['DEATH_TIME_DISCH'] = pd.to_timedelta(cohort_df['DEATH_TIME_DISCH'])
cohort_df['DEATH_TIME_DISCH'] = (cohort_df.DEATH_TIME_DISCH / pd.Timedelta(days = 1)).round(0)
cohort_df['DEATH_TIME_DISCH'] = cohort_df['DEATH_TIME_DISCH'] + 1

In [14]:
cohort_df['DOB'] = pd.to_datetime(cohort_df['DOB'], format= "%Y-%m-%d")
cohort_df['DOB'] = cohort_df.DOB.dt.year
cohort_df['ADMITTYEAR'] = pd.to_datetime(cohort_df['ADMITTIME'], format= "%Y-%m-%d")
cohort_df['ADMITTYEAR'] = cohort_df.ADMITTYEAR.dt.year
cohort_df['AGE'] = cohort_df['ADMITTYEAR'] - cohort_df['DOB']
cohort_df.drop(columns=['ADMITTYEAR', 'DOB', 'DOD'], inplace=True)

In [15]:
cohort_df = cohort_df[['SUBJECT_ID', 'HADM_ID', 'GENDER', 'AGE', 'ETHNICITY', 'DIAGNOSIS', 'ADMITTIME', 
                       'DISCHTIME', 'DEATHTIME', 'HOSPITAL_EXPIRE_FLAG', 'DEATH_TIME_DISCH', 'EXPIRE_FLAG']]

In [16]:
cohort_df.head(3)

In [17]:
print(cohort_df.SUBJECT_ID.nunique())
print(cohort_df.HADM_ID.nunique())

46520
58976


### icustays

In [18]:
icustays = pd.read_csv(mimiciii + "ICUSTAYS.csv")

In [19]:
icustays.drop(columns=['ROW_ID', 'DBSOURCE', 'FIRST_CAREUNIT', 'LAST_CAREUNIT', 
                       'FIRST_WARDID', 'LAST_WARDID'], inplace=True)

In [20]:
icustays['INTIME']  = pd.to_datetime(icustays['INTIME'])
icustays['OUTTIME'] = pd.to_datetime(icustays['OUTTIME'])
icustays['LOS'] = icustays['LOS'].round(1)

In [21]:
icustays.head(2)

In [22]:
print(icustays.SUBJECT_ID.nunique())
print(icustays.HADM_ID.nunique())
print(icustays.ICUSTAY_ID.nunique())

46476
57786
61532


### merge cohort and icu

In [23]:
cohort_df = pd.merge(cohort_df, icustays, on=['SUBJECT_ID', 'HADM_ID'], how='left')

In [24]:
cohort_df['ADM_IN_DIFF'] = cohort_df['INTIME'] - cohort_df['ADMITTIME']
cohort_df['ADM_IN_DIFF'] = pd.to_timedelta(cohort_df['ADM_IN_DIFF'])
cohort_df['ADM_IN_DIFF'] = (cohort_df.ADM_IN_DIFF / pd.Timedelta(hours = 1)).round(1)

cohort_df['OUT_DIS_DIFF'] = cohort_df['DISCHTIME'] - cohort_df['OUTTIME']
cohort_df['OUT_DIS_DIFF'] = pd.to_timedelta(cohort_df['OUT_DIS_DIFF'])
cohort_df['OUT_DIS_DIFF'] = (cohort_df.OUT_DIS_DIFF / pd.Timedelta(hours = 1)).round(1)

In [25]:
cohort_df = cohort_df[(cohort_df.ADM_IN_DIFF > -24)]
cohort_df = cohort_df[~(((cohort_df.OUT_DIS_DIFF < -24) & (cohort_df.HOSPITAL_EXPIRE_FLAG == 0)))]

condition_1 = (cohort_df.ADM_IN_DIFF < 0)
cohort_df.loc[condition_1, 'ADMITTIME']  = cohort_df.loc[condition_1, 'INTIME']

condition_2 = ((cohort_df.OUT_DIS_DIFF < 0) & (cohort_df.HOSPITAL_EXPIRE_FLAG == 0))
cohort_df.loc[condition_2, 'DISCHTIME']  = cohort_df.loc[condition_2, 'OUTTIME']

cohort_df = cohort_df[cohort_df.OUT_DIS_DIFF > -48]

condition_3 = (cohort_df.OUT_DIS_DIFF < 0)
cohort_df.loc[condition_3, 'OUTTIME']  = cohort_df.loc[condition_3, 'DISCHTIME']

In [26]:
cohort_df.drop(columns=['OUT_DIS_DIFF', 'ADM_IN_DIFF'], inplace=True)

In [27]:
cohort_df = cohort_df[(cohort_df.ICUSTAY_ID.notnull()) & (cohort_df.INTIME >= cohort_df.ADMITTIME) & 
                      (cohort_df.OUTTIME <= cohort_df.DISCHTIME)]

In [28]:
cohort_df['ICU_EXPIRE_FLAG'] = 0
cohort_df.loc[((cohort_df['DEATHTIME'] > cohort_df['INTIME']) & (cohort_df['DEATHTIME'] < cohort_df['OUTTIME'] 
                                                              + pd.Timedelta(hours=6))), 'ICU_EXPIRE_FLAG'] = 1

In [29]:
cohort_df['ICU_LOS_H'] = cohort_df['OUTTIME'] - cohort_df['INTIME']
cohort_df['ICU_LOS_H'] = pd.to_timedelta(cohort_df['ICU_LOS_H'])

cohort_df['ICU_LOS_H'] = (cohort_df.ICU_LOS_H / pd.Timedelta(hours = 1)).round(1)
cohort_df.rename(columns={"LOS": "ICU_LOS_D"}, inplace=True)

In [30]:
cohort_df['HOSP_LOS'] = cohort_df['DISCHTIME'] - cohort_df['ADMITTIME']
cohort_df['HOSP_LOS'] = pd.to_timedelta(cohort_df['HOSP_LOS'])

cohort_df['HOSP_LOS_D'] =  (cohort_df.HOSP_LOS / pd.Timedelta(days = 1)).round(1)
cohort_df['HOSP_LOS_H'] =  (cohort_df.HOSP_LOS / pd.Timedelta(hours= 1)).round(1)

cohort_df.drop(columns=['HOSP_LOS'], inplace=True)

In [31]:
cohort_df = cohort_df[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'GENDER', 'AGE', 'ETHNICITY', 'INTIME', 'OUTTIME',
                       'ADMITTIME', 'DISCHTIME', 'ICU_LOS_H', 'ICU_LOS_D', 'HOSP_LOS_H', 'HOSP_LOS_D',
                       'ICU_EXPIRE_FLAG', 'HOSPITAL_EXPIRE_FLAG', 'DEATHTIME', 'DEATH_TIME_DISCH',
                       'EXPIRE_FLAG']]

In [32]:
cohort_df.head(3)

In [33]:
print(cohort_df.SUBJECT_ID.nunique())
print(cohort_df.HADM_ID.nunique())
print(cohort_df.ICUSTAY_ID.nunique())
print(cohort_df[cohort_df.DEATH_TIME_DISCH.notnull()].ICUSTAY_ID.nunique())

45300
56143
59801
23461


### Split for Extraction

In [34]:
def split_list(original_list, num_splits):
    
    split_size = len(original_list) // num_splits
    split_lists = [[] for _ in range(num_splits)]
    
    for i, item in enumerate(original_list):
        index = i // split_size
        if index >= num_splits:
            index = num_splits - 1
        split_lists[index].append(item)

    return split_lists

In [35]:
original_list = list(cohort_df.SUBJECT_ID.unique())
split_lists = split_list(original_list, 3)

In [36]:
for i, lst in enumerate(split_lists):
    print(f"List {i+1} size: {len(lst)}")

List 1 size: 15100
List 2 size: 15100
List 3 size: 15100


In [37]:
chr1 = split_lists[0]
chr2 = split_lists[1]
chr3 = split_lists[2]

In [38]:
cohort1 = cohort_df[cohort_df.SUBJECT_ID.isin(chr1)].copy()
cohort2 = cohort_df[cohort_df.SUBJECT_ID.isin(chr2)].copy()
cohort3 = cohort_df[cohort_df.SUBJECT_ID.isin(chr3)].copy()

In [39]:
print(cohort1.ICUSTAY_ID.nunique())
print(cohort2.ICUSTAY_ID.nunique())
print(cohort3.ICUSTAY_ID.nunique())

20218
20232
19351


In [40]:
cohort1_subject_id = list(cohort1.SUBJECT_ID.unique())
cohort1_hadm_id = list(cohort1.HADM_ID.unique())
cohort1_stay_id = list(cohort1.ICUSTAY_ID.unique())

cohort2_subject_id = list(cohort2.SUBJECT_ID.unique())
cohort2_hadm_id = list(cohort2.HADM_ID.unique())
cohort2_stay_id = list(cohort2.ICUSTAY_ID.unique())

cohort3_subject_id = list(cohort3.SUBJECT_ID.unique())
cohort3_hadm_id = list(cohort3.HADM_ID.unique())
cohort3_stay_id = list(cohort3.ICUSTAY_ID.unique())

In [41]:
cohort1_stay_id = [int(id) for id in cohort1_stay_id]
cohort2_stay_id = [int(id) for id in cohort2_stay_id]
cohort3_stay_id = [int(id) for id in cohort3_stay_id]

### Save Cohorts

In [42]:
with open("../Data/Cohort/cohort1_subject_id.txt", "w") as f:
    for subject_id in cohort1_subject_id:
        f.write(str(subject_id) +"\n")
        
with open("../Data/Cohort/cohort1_hadm_id.txt", "w") as f:
    for hadm_id in cohort1_hadm_id:
        f.write(str(hadm_id) +"\n")
        
with open("../Data/Cohort/cohort1_stay_id.txt", "w") as f:
    for stay_id in cohort1_stay_id:
        f.write(str(stay_id) +"\n")

In [43]:
with open("../Data/Cohort/cohort2_subject_id.txt", "w") as f:
    for subject_id in cohort2_subject_id:
        f.write(str(subject_id) +"\n")
        
with open("../Data/Cohort/cohort2_hadm_id.txt", "w") as f:
    for hadm_id in cohort2_hadm_id:
        f.write(str(hadm_id) +"\n")
        
with open("../Data/Cohort/cohort2_stay_id.txt", "w") as f:
    for stay_id in cohort2_stay_id:
        f.write(str(stay_id) +"\n")

In [44]:
with open("../Data/Cohort/cohort3_subject_id.txt", "w") as f:
    for subject_id in cohort3_subject_id:
        f.write(str(subject_id) +"\n")
        
with open("../Data/Cohort/cohort3_hadm_id.txt", "w") as f:
    for hadm_id in cohort3_hadm_id:
        f.write(str(hadm_id) +"\n")
        
with open("../Data/Cohort/cohort3_stay_id.txt", "w") as f:
    for stay_id in cohort3_stay_id:
        f.write(str(stay_id) +"\n")

### Selective Cohort Selection

### note

In [107]:
note = pd.read_csv(mimiciii + "NOTEEVENTS.csv")

<ipython-input>:1: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  note = pd.read_csv(mimiciii + "NOTEEVENTS.csv")


In [108]:
note = note[['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'CATEGORY', 'CHARTDATE']]
cohort_copy = cohort_df.copy()
cohort_copy = cohort_copy[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'INTIME', 'OUTTIME']].reset_index(drop=True)
note_in_icu = pd.merge(note, cohort_copy, on=['SUBJECT_ID', 'HADM_ID'], how='left')
note_in_icu = note_in_icu[((note_in_icu.CHARTDATE >= note_in_icu.INTIME) &
                           (note_in_icu.CHARTDATE <= note_in_icu.OUTTIME))]

In [109]:
note_in_icu = note_in_icu[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ROW_ID', 'CATEGORY', 'CHARTDATE']]
note_in_icu.rename(columns={"ROW_ID": "NOTE_ID"}, inplace=True)
note_in_icu.reset_index(inplace=True, drop=True)
note_in_icu = note_in_icu[note_in_icu.CATEGORY != 'Discharge summary']

In [112]:
note_in_icu.head(3)

In [113]:
print(note_in_icu.ICUSTAY_ID.nunique())
print(note_in_icu.shape)

41330
(662775, 7)


### Filter based on Note Reports 

In [79]:
stay_id_report_list = list(note_in_icu.ICUSTAY_ID.unique())

In [80]:
cohort_df = cohort_df[cohort_df.ICUSTAY_ID.isin(stay_id_report_list)]

In [81]:
print(cohort_df.SUBJECT_ID.nunique())
print(cohort_df.HADM_ID.nunique())
print(cohort_df.ICUSTAY_ID.nunique())

32336
39763
42151


### Filter based on Age

In [82]:
min_age = 20
max_age = 75
cohort_df = cohort_df[(cohort_df.AGE > min_age) & (cohort_df.AGE < max_age)]

In [83]:
print(cohort_df.SUBJECT_ID.nunique())
print(cohort_df.HADM_ID.nunique())
print(cohort_df.ICUSTAY_ID.nunique())

20781
25914
27574


### Filter based on ICU length of stay

In [84]:
min_los = 24

cohort_df = cohort_df[cohort_df.ICU_LOS_H > min_los]

In [85]:
print(cohort_df.SUBJECT_ID.nunique())
print(cohort_df.HADM_ID.nunique())
print(cohort_df.ICUSTAY_ID.nunique())

18458
22679
24088


### Filter based on first Hospital and ICU Admission

In [86]:
cohort_df.sort_values(by=['SUBJECT_ID', 'ADMITTIME', 'INTIME'], inplace=True)

In [87]:
cohort_df = cohort_df.groupby(['SUBJECT_ID']).head(1)
cohort_df['ICUSTAY_ID'] = cohort_df['ICUSTAY_ID'].astype(int)

In [89]:
print(cohort_df.SUBJECT_ID.nunique())
print(cohort_df.HADM_ID.nunique())
print(cohort_df.ICUSTAY_ID.nunique())

18458
18458
18458


### Final Admission ID 

In [90]:
final_subject_id = list(cohort_df.SUBJECT_ID.unique())
final_hadm_id = list(cohort_df.HADM_ID.unique())
final_stay_id = list(cohort_df.ICUSTAY_ID.unique())

### Write patients admission ids

In [91]:
# with open("../Data/Cohort/subject_id_cohort.txt", "w") as f:
#     for subject_id in final_subject_id:
#         f.write(str(subject_id) +"\n")

In [92]:
# with open("../Data/Cohort/hadm_id_cohort.txt", "w") as f:
#     for hadm_id in final_hadm_id:
#         f.write(str(hadm_id) +"\n")

In [93]:
# with open("../Data/Cohort/stay_id_cohort.txt", "w") as f:
#     for stay_id in final_stay_id:
#         f.write(str(stay_id) +"\n")